<div class='alert alert-warning'>

# JupyterLite warning

- Running the **scikit-plots 0.5.dev0** interactive examples in JupyterLite is experimental and may not always work as expected.
- With high load times especially on low-resource platforms, and the version of scikit-plots might not be in sync with the one you are browsing the documentation for.
- If you encounter any issues, please report them on the [scikit-plots issue tracker](https://github.com/scikit-plots/scikit-plots/issues/new/choose).
- `micropip/piplite/pip` use `%pip` in JupyterLite instead of `pip` or `!pip`

```python
## Installing the dependencies first, and then scikit-plots from Anaconda.org.
import piplite; await piplite.install(  # or micropip
   'scikit-plots==0.5.dev0',            # Download scikit-plots *pyodide_20XX_0_wasm32.whl
   index_urls='https://pypi.anaconda.org/scikit-plots-wheels-staging-nightly/simple',
); import sklearn; import scikitplot as sp; sp.show_versions();
 ```

</div>

Direct pipeline control:


In [ ]:
from pathlib import Path
from scikitplot.corpus import CorpusPipeline, ParagraphChunker
pipeline = CorpusPipeline(chunker=ParagraphChunker())
result = pipeline.run(Path("article.txt"))  # doctest: +SKIP
print(f"{result.n_documents} chunks from {result.source}")  # doctest: +SKIP

The dependency-free default sentence backend is ``REGEX``. Passing a spaCy
model name explicitly selects the spaCy shorthand instead:


In [ ]:
from scikitplot.corpus import SentenceChunker
portable = SentenceChunker()
spacy_chunker = SentenceChunker("en_core_web_sm")  # doctest: +SKIP

High-level builder:


In [ ]:
from scikitplot.corpus import CorpusBuilder, BuilderConfig
builder = CorpusBuilder(
    BuilderConfig(
        chunker="paragraph",
        normalize=True,
        enrich=True,
        build_index=True,
    )
)
result = builder.build("./data/")  # doctest: +SKIP
results = builder.search("quantum computing")  # doctest: +SKIP

Reusable declarative configuration:


In [ ]:
from scikitplot.corpus import FluentCorpus
fluent = FluentCorpus().chunker("paragraph").storage("memory")
fluent.validate()

[]

Materialization is explicit and performs no source read by itself:


In [ ]:
from scikitplot.corpus import RuntimePolicy
with fluent.materialize(policy=RuntimePolicy(allow_network=False)) as runtime:
    len(runtime.documents)

0

A configured source becomes operational only when ``run()`` is called:


In [ ]:
runtime_fluent = (
    FluentCorpus().source("article.txt").chunker("paragraph").storage("memory")
)
with runtime_fluent.materialize() as runtime:
    result = runtime.run()  # doctest: +SKIP

Network and optional-capability examples:

Live URLs, OCR, ASR, spaCy, NLTK resource-backed NLP, model embeddings, and
native vector backends depend on the corresponding environment capability.
User-facing examples should not fabricate results when a capability is absent.
Documentation/gallery examples should either use a portable executed path or
report a clear skip while keeping the optional configuration visible.

URL ingestion:


In [ ]:
result = pipeline.run_url("https://en.wikipedia.org/wiki/Python")  # doctest: +SKIP

YouTube transcript:


In [ ]:
result = pipeline.run(
    "https://www.youtube.com/watch?v=rwPISgZcYIk"
)  # doctest: +SKIP

Image OCR:


In [ ]:
from scikitplot.corpus import DocumentReader
reader = DocumentReader.create(Path("scan.png"))  # doctest: +SKIP
docs = list(reader.get_documents())  # doctest: +SKIP

With model embeddings:


In [ ]:
from scikitplot.corpus import EmbeddingEngine
engine = EmbeddingEngine(backend="sentence_transformers")  # doctest: +SKIP

Dependency-free local helpers:


In [ ]:
from scikitplot.corpus import HAMLET_TEXT, HashEmbedder, SimpleEnricherSpec
HashEmbedder(dimension=32)([HAMLET_TEXT[:120]]).shape

(1, 32)

In [ ]:
FluentCorpus().enricher(SimpleEnricherSpec()).validate()

[]

``HashEmbedder`` is a deterministic lexical hashing baseline, not a learned
semantic model. ``HAMLET_TEXT`` is bundled convenience sample data rather than
an authoritative scholarly edition.

See ``scikitplot/corpus/README.md`` for the user-oriented API map, runtime
policy boundary, retrieval modes, and optional-capability guidance.